# 1. Problem Tanımı ve Proje Amacı

## Problem

Modern not uygulamaları (örneğin Apple Notes, Notion, Obsidian vb.) 
notları doğrusal bir liste halinde saklar.

Bu sistem bir arşivdir — fakat bir hafıza değildir.

Anahtar kelime araması yalnızca kelime eşleşmesi yapar.  
Kavramsal ve bağlamsal yakınlıkları algılayamaz.

Sonuç:

- Notlar arası gizli bağlantılar kaybolur.
- Kullanıcı kendi bilgisini yeniden keşfedemez.
- Dijital hafıza pasif bir depoya dönüşür.

---

## Amaç

Bu proje, notları anahtar kelimelere göre değil  
**anlamsal yakınlıklarına göre analiz eden bir sistem** geliştirmeyi amaçlar.

Sistem:

- Notları vektör temsillere dönüştürür,
- Yoğunluk tabanlı kümeleme uygular,
- Notlar arası bağlantı dokusunu ortaya çıkarır.

Bu çalışma bir arama sistemi değil,  
bir **Anlamsal Hafıza Motoru** tasarımıdır.

# 2. Ortam Kurulumu ve Gerekli Kütüphanelerin Yüklenmesi

Bu çalışma, anlamsal temsil üretimi, boyut indirgeme, kümeleme ve isimlendirme işlemleri için aşağıdaki Python kütüphanelerini kullanmaktadır:

- `sentence-transformers` → SBERT tabanlı embedding üretimi  
- `umap-learn` → Boyut indirgeme  
- `hdbscan` → Yoğunluk tabanlı kümeleme  
- `scikit-learn` → Benzerlik ölçümü ve değerlendirme metrikleri  
- `networkx` → Bilgi grafiği oluşturma  
- `matplotlib` → Görselleştirme  
- `nltk` → Doğal dil işleme (POS tagging ve stopword filtreleme)

Gerekli paketler kurulu değilse aşağıdaki komut çalıştırılmalıdır:

In [1]:
%pip install sentence-transformers umap-learn hdbscan networkx matplotlib scikit-learn nltk seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Veri işleme
import pandas as pd
import numpy as np

# Metin işleme
import re

# NLP
import nltk
from nltk.corpus import stopwords

# Vektörleştirme
from sentence_transformers import SentenceTransformer

# Boyut indirgeme
import umap.umap_ as umap

# Kümeleme
import hdbscan

# Benzerlik ölçümü
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns

# Bilgi grafiği
import networkx as nx

# Notebook ayarları
%matplotlib inline
plt.style.use("seaborn-v0_8")

# 3. Keşifsel Veri Analizi (EDA)

Bu aşamada veri seti yüklenir ve detaylı incelenir. Amaç, modelleme öncesi veri kalitesini görmek ve olası sorunları tespit etmektir.  
Analizler şunları kapsar:

- Not uzunlukları ve dağılımları
- Saf gürültü (yalnızca URL veya sayı) tespiti
- Görselleştirme ile dağılım analizi

In [ ]:
# Dataset'i yükle
df = pd.read_csv("dataset.csv")

# Uzunluk ve saf gürültü hesapları
df["word_length"] = df["text"].astype(str).apply(lambda x: len(x.split()))
df["char_length"] = df["text"].astype(str).apply(len)
df["is_pure_noise"] = df["text"].apply(lambda x: bool(re.fullmatch(r"https?://\S+", x.strip()) or re.fullmatch(r"\d+", x.strip())))

# Özet bilgiler
print(f"Toplam Not: {len(df)}")
print(f"En Kısa Not: {df['word_length'].min()} kelime / {df['char_length'].min()} karakter")
print(f"Ortalama Not: {df['word_length'].mean():.1f} kelime / {df['char_length'].mean():.0f} karakter")
print(f"En Uzun Not: {df['word_length'].max()} kelime / {df['char_length'].max()} karakter")

# Saf gürültü
noise_count = df['is_pure_noise'].sum()
print(f"\nSaf Gürültü Sayısı: {noise_count} ({noise_count/len(df)*100:.2f}%)")

# Saf gürültü örnekleri
if noise_count > 0:
    noise_example = df[df["is_pure_noise"]].head(1)["text"].values[0]
    print("Örnek Saf Gürültü Notu:", noise_example)

# Kelime sayısı dağılım grafiği
plt.figure(figsize=(7,4))
sns.histplot(df["word_length"], bins=25, kde=True, color="teal", alpha=0.7)
plt.title("Not Kelime Sayısı Dağılımı")
plt.xlabel("Kelime Sayısı")
plt.ylabel("Frekans")
plt.tight_layout()
plt.show()

# 4. Veri Temizliği ve Ön İşleme

Bu aşamada notlar modele verilmeden önce temizlenir ve düzenlenir.  
Amaç, gereksiz verileri çıkarmak ve tüm metni standart bir forma getirmektir.

Bu adımda:

- Sadece link veya sadece sayı içeren notlar (saf gürültü) silinir  
- 5 karakterden kısa notlar çıkarılır  
- Metinlerin başındaki ve sonundaki boşluklar temizlenir  
- Fazla boşluklar teke indirilir  
- Tüm harfler küçük harfe dönüştürülür  
- Aynı olan tekrar eden notlar (duplicate) kaldırılır  

Bu işlemler sayesinde veri daha temiz, daha tutarlı ve modele hazır hâle gelir.

In [ ]:
# Veri setindeki toplam başlangıç not sayısını alıyoruz
initial_count = len(df)

# Orijinal veriyi bozmamak için kopya oluşturuyoruz
df_clean = df.copy()

# 1) SADECE link veya sadece sayı olan satırları "saf gürültü" kabul edip siliyoruz
noise_removed = df_clean[df_clean["is_pure_noise"]]
df_clean = df_clean[~df_clean["is_pure_noise"]]

# 2) 5 karakterden kısa notları anlamsız kabul edip siliyoruz (örn: "ok", "hi")
short_removed = df_clean[df_clean["char_length"] < 5]
df_clean = df_clean[df_clean["char_length"] >= 5]

# 3) Metni standart hale getiriyoruz:
#    - Baştaki ve sondaki boşlukları siliyoruz
#    - Birden fazla boşluğu teke indiriyoruz
#    - Tüm harfleri küçültüyoruz
df_clean["text"] = (
    df_clean["text"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# 4) Birebir aynı olan tekrar eden notları (duplicate) temizliyoruz
duplicate_removed = df_clean[df_clean.duplicated(subset="text")]
df_clean = df_clean.drop_duplicates(subset="text")

# Temizlik sonrası kalan toplam not sayısı
final_count = len(df_clean)

# Modelde kullanacağımız son temiz not listesi
notes = df_clean["text"].tolist()

# Son durum
print("Başlangıç Not Sayısı:", initial_count)
print("Temizlik Sonrası Not Sayısı:", final_count)
print("Toplam Silinen Not:", initial_count - final_count)

print("\nSilinen Saf Gürültü Sayısı:", len(noise_removed))
if len(noise_removed) > 0:
    print("Örnek Saf Gürültü:", noise_removed["text"].iloc[0])

print("\nSilinen Kısa Not Sayısı:", len(short_removed))
if len(short_removed) > 0:
    print("Örnek Kısa Not:", short_removed["text"].iloc[0])

print("\nSilinen Duplicate Sayısı:", len(duplicate_removed))
if len(duplicate_removed) > 0:
    print("Örnek Duplicate:", duplicate_removed["text"].iloc[0])

# 5. Embedding Üretimi (SBERT)

Bu aşamada temizlenmiş notları yapay zekânın anlayabileceği sayısal forma dönüştürüyoruz.

Bilgisayarlar metni doğrudan anlayamaz.  
Bu yüzden her notu sabit uzunlukta bir sayı dizisine (vektöre) çeviriyoruz.  
Bu sayısal temsile **embedding** denir.

Bu projede:

- **Model:** `all-MiniLM-L6-v2`  
  (Hızlı ve hafif bir SBERT modeli)

- Her not **384 boyutlu bir vektöre** dönüştürülür  
- Tüm notlar aynı uzunlukta temsil edilir  
- Benzer notlar sayısal olarak birbirine daha yakın olur  
- Farklı notlar daha uzak olur  

Embedding üretiminde ayrıca vektörler normalize edilir.  
Bu sayede karşılaştırma yapılırken uzunluk değil, **anlamsal yön** dikkate alınır.

Embeddingler sayesinde artık:

- Notlar arası benzerlik hesaplanabilir  
- Kümeleme algoritmaları uygulanabilir  
- Anlamsal yakınlık matematiksel olarak ölçülebilir  

Bu adım, sistemi anahtar kelime eşleşmesinden çıkarıp  
metni gerçek anlamda **anlam tabanlı** olarak karşılaştırabilir hale getirir.

In [ ]:
# SBERT modeli yükle
model = SentenceTransformer("all-MiniLM-L6-v2")

# Temizlenmiş not listesi
texts = notes  # Önceki adımda clean_notes() ile elde edildi

# Embedding üretimi
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Bilgi çıktısı
print(f"Toplam not: {len(texts)}")
print("Embedding boyutu:", embeddings.shape[1])  # Embedding boyutu

# 6. Boyut İndirgeme (UMAP)

SBERT ile her notu 384 boyutlu sayısal bir temsile dönüştürdük.  
Ancak bu kadar yüksek boyutlu veriyle doğrudan çalışmak hem zor hem de gereksiz karmaşıktır.

Bu nedenle UMAP kullanarak boyutu 20'ye düşürüyoruz.

Amaç:

- Hesaplamayı hızlandırmak  
- Kümeleme işlemini daha stabil hâle getirmek  
- Benzer notların birbirine yakın kalmasını sağlamak  

Bu işlem sırasında:

- Her not daha düşük boyutlu bir temsile indirgenir  
- Anlamsal yakınlık korunur  
- Veri daha sade bir yapıya kavuşur  

Sonuç olarak, embeddingler kümeleme için daha uygun bir forma dönüşür.

In [ ]:
reducer = umap.UMAP(
    n_neighbors=10,      # Her not için dikkate alınan komşu sayısı
    n_components=20,     # Yeni hedef boyut (384 → 20)
    min_dist=0.05,       # Noktaların birbirine ne kadar yakın olabileceğini kontrol eder
    metric="cosine",     # SBERT embeddingleri için uygun mesafe türü
    random_state=42      # Sonuçların her çalıştırmada aynı çıkması için
)

# Embeddingleri daha düşük boyuta indiriyoruz
umap_embeddings = reducer.fit_transform(embeddings)

# Sonuç bilgisi
print("Her not için yeni boyut:", umap_embeddings.shape[1])

# 7. HDBSCAN ile Kümeleme

Bu aşamada benzer notları otomatik olarak grupluyoruz.

HDBSCAN adlı algoritma, notların yoğunluk yapısına bakarak
hangi notların birlikte gruplanması gerektiğini belirler.

Önemli nokta:

- Benzer notlar aynı cluster içine düşer
- Alakasız veya tek başına kalan notlar “gürültü” olarak işaretlenir (-1)

Bu yöntem sayesinde:

- Önceden cluster sayısını belirlemek zorunda kalmayız
- Veri kendi doğal yapısına göre gruplanır
- Anlamsal olarak yakın notlar birlikte toplanır

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=4,   # Bir kümede en az kaç not olmalı
    min_samples=2,        # Yoğunluk hesabı için komşu sayısı
    metric="euclidean",   # UMAP sonrası uygun mesafe ölçümü
    cluster_selection_method="eom",
    prediction_data=True
)

# Kümeleme işlemi
cluster_labels = clusterer.fit_predict(umap_embeddings)

# Sonuçları tabloya ekliyoruz
df_clusters = pd.DataFrame({
    "note": texts,
    "cluster": cluster_labels
})

# Cluster dağılımı
print("Cluster dağılımı:")
print(df_clusters["cluster"].value_counts().sort_index())

# Noise Oranı ve Başarı Analizi
total = len(df_clusters)
noise = len(df_clusters[df_clusters["cluster"] == -1])
noise_ratio = noise / total * 100
print("\nCluster dışı (noise):", noise)
print(f"Noise Oranı: {noise_ratio:.2f}%")

# 8. Küme İsimlendirme (Akıllı Klasörleme)

Kümeleme sonrası sistem benzer notları grupladı.  
Ancak bu gruplar sadece numaralardan oluşuyor (Cluster 0, 1, 2...).

Bu aşamada:

- Her cluster’ın içeriği analiz edilir
- İçindeki kelimelere göre anlamlı bir isim üretilir
- Benzer cluster isimlerinin çakışması engellenir
- Gürültü (noise) notlar ayrı bir klasöre alınır

İsimlendirme iki aşamalı çalışır:

1. Önceden belirlenen kategori sözlüğü kontrol edilir  
2. Eğer eşleşme yoksa TF-IDF ile en baskın kelimeler bulunur  

Sonuç olarak, bu adım teknik çıktıyı insanın anlayabileceği bir yapıya dönüştürür.

In [ ]:
# Kategori sözlüğü (Önceden tanımlı domain anahtar kelimeleri ile cluster isimlendirmede rule-based öncelik sağlar)
CATEGORY_MAP = {

    # Software ile ilgili teknik terimler
    'CODING & DEV': [
        'git','github','branch','merge','commit','docker','api','backend',
        'frontend','server','bug','debug','regex','python','java','exception',
        'nullpointer','ssh','terminal','cli','script','function','variable',
        'df','pandas','numpy','tokenizer','huggingface'
    ],

    # Güvenlik ve sistem yönetimi terimleri
    'SECURITY & ADMIN': [
        'password','wifi','admin','pass','pin','router','login',
        'auth','authentication','firewall','ip','port','ssh','key'
    ],

    # Akademik ve araştırma bağlamlı kelimeler
    'ACADEMIC & RESEARCH': [
        'paper','thesis','arxiv','research','citation','prof',
        'study','experiment','dataset','attention','transformer',
        'theory','analysis','publication'
    ],

    # Felsefi ve kavramsal içerik
    'PHILOSOPHY & IDEAS': [
        'consciousness','meaning','entropy','existence',
        'wittgenstein','simulacra','memory','time','silence',
        'fear','identity','mind','thought','being'
    ],

    # Ses ve müzik ile ilgili terimler
    'AUDIO & SOUND': [
        'audio','sound','music','silence','noise','plugin',
        'sample','mix','track','birdsongs','shazam'
    ],

    # Market ve alışveriş notları
    'GROCERY & SHOPPING': [
        'milk','eggs','bread','pasta','oil','chicken','tomatoes',
        'tomato','cheese','yogurt','butter','rice','onion','garlic',
        'pepper','salt','apple','banana','meat','fish','shopping',
        'market','grocery','buy','supermarket'
    ],

    # Günlük rutin ve kişisel işler
    'PERSONAL & ROUTINE': [
        'routine','reset','room','clean','dentist','gym',
        'workout','health','sleep','habit'
    ],

    # Bağlantı ve referans içerikleri
    'LINKS & REFERENCES': [
        'http','https','link','url','github.com',
        'arxiv.org','youtube','drive','doc'
    ]
}

# TF-IDF fallback aşamasında kullanılacak stopword listesi.
custom_stop_words = list(stopwords.words('english'))


# Duplicate Name Handler (Tekrarlayan isimleri engellemek için gerekirse ayırt edici kelime ekleyerek benzersiz klasör adı üretir.)
def resolve_duplicate_name(base_name, text_list, used_names):

    final_name = base_name
    counter = 1

    while final_name in used_names:

        try:
            vec = TfidfVectorizer(stop_words=custom_stop_words, ngram_range=(1,1))
            tfidf = vec.fit_transform(text_list)
            scores = tfidf.sum(axis=0).A1
            feature_names = vec.get_feature_names_out()
            sorted_idx = scores.argsort()[::-1]

            distinguishing_word = None

            for idx in sorted_idx:
                word = feature_names[idx].upper()
                if word not in base_name.replace("&","").split():
                    distinguishing_word = word
                    break

            final_name = f"{base_name} ({distinguishing_word})" if distinguishing_word else f"{base_name} ({counter})"

        except:
            final_name = f"{base_name} ({counter})"

        counter += 1

    used_names.add(final_name)
    return final_name


# Cluster içeriğine göre en uygun klasör adını üretir: önce rule-based kategori, yeterli eşleşme yoksa TF-IDF fallback uygulanır.
def get_best_name(text_list, used_names):

    combined_text = " ".join(text_list).lower()
    words = re.sub(r"[^\w\s]", "", combined_text).split()

    scores = {cat: 0 for cat in CATEGORY_MAP}

    for word in words:
        for cat, keywords in CATEGORY_MAP.items():
            if word in keywords:
                scores[cat] += 1

    best_cat = max(scores, key=scores.get)

    if scores[best_cat] >= 3:
        base_name = best_cat
    else:
        vectorizer = TfidfVectorizer(stop_words=custom_stop_words, max_features=2)
        tfidf = vectorizer.fit_transform([combined_text])
        keywords = vectorizer.get_feature_names_out()
        base_name = " & ".join([w.upper() for w in keywords]) if len(keywords) > 0 else "MISC"

    return resolve_duplicate_name(base_name, text_list, used_names)


# Her cluster için isim üretir, noise (-1) ayrı etiketlenir.
cluster_names = {}
used_folder_names = set()

unique_clusters = sorted(df_clusters["cluster"].unique())

for cluster_id in unique_clusters:
    if cluster_id == -1:
        continue

    cluster_docs = df_clusters[df_clusters["cluster"] == cluster_id]["note"].tolist()
    cluster_names[cluster_id] = get_best_name(cluster_docs, used_folder_names)

cluster_names[-1] = "NOISE"


# Üretilen cluster isimlerini ve içeriklerini detaylı şekilde yazdırır.
for cluster_id, folder_name in cluster_names.items():

    cluster_docs = df_clusters[df_clusters["cluster"] == cluster_id]["note"].tolist()

    print(f"\n📂 {folder_name}")
    print(f"Toplam Not: {len(cluster_docs)}\n")

    for i, note in enumerate(cluster_docs, 1):
        print(f"{i}. {note}")

# 9. 2D Görselleştirme (UMAP)

Kümeleme tamamlandıktan sonra embeddingleri 2 boyuta indirerek görselleştiriyoruz.

Yüksek boyutlu (384D) vektörleri doğrudan görselleştirmek mümkün olmadığı için,
**UMAP (Uniform Manifold Approximation and Projection)** kullanıyoruz.

Bu sayede:

- Anlamsal olarak benzer notlar birbirine yakın görünür  
- Farklı konular uzakta konumlanır  
- Cluster yapısı görsel olarak analiz edilebilir  
- Gürültü (noise) noktaları açıkça ayırt edilir  

Kullanılan parametreler:

- **n_neighbors=10** → Lokal komşuluk yapısını korur  
- **n_components=2** → 2 boyutlu projeksiyon  
- **min_dist=0.1** → Clusterların sıkışıklık seviyesi  
- **metric="cosine"** → SBERT embeddingleriyle uyumlu mesafe metriği  

Bu adım, kümeleme sonucunun görsel doğrulamasını sağlar.

In [ ]:
# 2D UMAP projeksiyonu: 384 boyutlu embeddingleri 2 boyuta indirir
viz_reducer = umap.UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = viz_reducer.fit_transform(embeddings)

# 2D koordinatları dataframe'e ekle
df_clusters["x"] = embedding_2d[:, 0]
df_clusters["y"] = embedding_2d[:, 1]

# Cluster isimlerini dataframe'e map et
df_clusters["cluster_name"] = df_clusters["cluster"].map(cluster_names)

# Her cluster için merkez (centroid) koordinatını hesapla
centroids = (
    df_clusters[df_clusters["cluster"] != -1]
    .groupby("cluster_name")[["x","y"]]
    .mean()
    .reset_index()
)

# Scatter plot oluştur
plt.figure(figsize=(12,8))

# Normal cluster noktalarını çiz
sns.scatterplot(
    data=df_clusters[df_clusters["cluster"] != -1],
    x="x",
    y="y",
    hue="cluster_name",
    palette="tab20",
    s=70
)

# Noise noktalarını ayrı ve belirgin şekilde göster
sns.scatterplot(
    data=df_clusters[df_clusters["cluster"] == -1],
    x="x",
    y="y",
    color="black",
    marker="X",
    s=60,
    alpha=0.7,
    label="NOISE"
)

# Her cluster merkezine kutulu etiket yerleştir
for _, row in centroids.iterrows():
    plt.text(
        row["x"],
        row["y"],
        row["cluster_name"],
        fontsize=9,
        weight="bold",
        ha="center",
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            boxstyle="round,pad=0.3"
        )
    )

plt.title("Not Kümeleme Sistemi: Dinamik Bilgi Grafiği")
plt.xlabel("")
plt.ylabel("")
plt.legend(title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()